# P001 — admission perfusion baseline

This notebook runs one frozen exploratory baseline on 99 eligible cases.
Use a CPU runtime. Cross-family review must be committed before Run All can
pass the authority gate. Keep all private outputs in your own Drive; do not
upload patient files to Git. No follow-ups run here. See SPEC.md and RESULT_CARD.md.

Set ARCHIVE to your existing checksum-pinned local train.7z, or DATA_ROOT to
an existing selectively staged tree. Reading the 99 GB archive directly from
Drive can be slow; existing verified local staging is preferable. The script
extracts only the 198 selected files. It never downloads or extracts the full
cohort. Return aggregate outputs plus original console and private audit
checkpoints through the agreed private channel; no automatic push occurs.

In [ ]:
from pathlib import Path
import subprocess, sys
PIN = '45c4a5d3329b210eea4a3e1b11027b1bca261fe5'
REPO = Path('/content/scout-pilot-' + PIN[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Moroseui/concept-research-scout.git', str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', PIN], cwd=REPO, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() == PIN
assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=REPO,text=True).strip(), 'Modified code; use a fresh checkout'
sys.path.insert(0,str(REPO))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT = Path('/content/drive/MyDrive/isles-pilot/P001-v1')
ARCHIVE = Path('/content/train.7z')
DATA_ROOT = None  # alternatively a verified selectively staged root


In [ ]:
EXPERIMENT = REPO/'campaigns/isles24-pilot/experiments/P001'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(EXPERIMENT/'requirements.txt')], check=True)
if not (EXPERIMENT/'review.json').is_file():
    raise RuntimeError('Opposing-family spec/code review pending; do not run patient workflow')


In [ ]:
from orchestrator.publication import run_logged
command = [sys.executable, str(EXPERIMENT/'run.py'), '--output-dir', str(OUTPUT)]
command += ['--data-root', str(DATA_ROOT)] if DATA_ROOT else ['--archive', str(ARCHIVE)]
exit_code = run_logged(command, OUTPUT)
print('Exit:', exit_code, '; console:', str(OUTPUT)+'.console.log')
if exit_code: raise RuntimeError('Attempt failed; evidence and checkpoints preserved')

In [ ]:
from orchestrator.publication import validate
policy = __import__('json').loads((EXPERIMENT/'publication.json').read_text())
validate(OUTPUT, policy)
print((OUTPUT/'RESULT_CARD.md').read_text())
print('Keep private audit files at:', str(OUTPUT)+'.private')